# Steps that turn the raw files into the dataframe we analyze:

## Dataset 1: Google Local Metadata

#### Step 1: Download and read dataset

- Download the file meta-Illinois.json.gz from Google Local Reviews Dataset - UCSD (https://cseweb.ucsd.edu/~jmcauley/datasets.html). Save it in EDA_Checkpoint/data/. 

- Code:

In [2]:
import pandas as pd
import numpy as np
import json
import gzip
from pathlib import Path

pd.set_option("display.max_columns", None)
data_dir = Path("../EDA_Checkpoint/data")

#2. Google local reviews

#Google Local Business Metadata
google_metadata = pd.read_json(
    data_dir/"meta-Illinois.json.gz",
    lines=True,
    compression="gzip"
)

#### Step 2: Filter to get only restaurants in Chicago

- Each row in dataset represents one business in Illinois. We need to filter it to get only businesses that are retaurants and are located in Chicago. You can first check the column names using .info().

- Code:

In [3]:
google_metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 179205 entries, 0 to 179204
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   name              179205 non-null  str    
 1   address           177042 non-null  str    
 2   gmap_id           179205 non-null  str    
 3   description       36668 non-null   str    
 4   latitude          179205 non-null  float64
 5   longitude         179205 non-null  float64
 6   category          178486 non-null  object 
 7   avg_rating        179205 non-null  float64
 8   num_of_reviews    179205 non-null  int64  
 9   price             37977 non-null   str    
 10  hours             139887 non-null  object 
 11  MISC              147528 non-null  object 
 12  state             127259 non-null  str    
 13  relative_results  166616 non-null  object 
 14  url               179205 non-null  str    
dtypes: float64(3), int64(1), object(4), str(7)
memory usage: 65.5+ MB


In [4]:
def has_restaurant_category(categories):
    return isinstance(categories, list) and any(
        "restaurant" in str(category).lower()
        for category in categories
    )

is_restaurant = google_metadata["category"].apply(has_restaurant_category)
is_chicago = google_metadata["address"].str.contains(
    r",\s*Chicago,\s*IL\b",
    case=False,
    na=False,
    regex=True,
)

chicago_restaurants = google_metadata.loc[
    is_restaurant & is_chicago
].copy()

print("Number of rows after filtering:", len(chicago_restaurants))

Number of rows after filtering: 8043


#### Step 3: Dropped Rows

- Check which restaurants appear more then once and drop them so there are no duplicates in dataset. You can first check gmap_id to see how many duplicated there are in dataset before removing them.

In [5]:
print("Number of duplicated before dropping them:",chicago_restaurants.duplicated(subset = "gmap_id").sum())

Number of duplicated before dropping them: 9


- Code:

In [6]:
chicago_restaurants = chicago_restaurants.drop_duplicates(
    subset="gmap_id",
    keep="first",
).copy()

print("Number of duplicated after dropping them:",chicago_restaurants.duplicated(subset = "gmap_id").sum())

Number of duplicated after dropping them: 0


#### Step 4: Type Fixes

- In this step we are changing the "state" column into string type and creating four groups: Permanently closed, Temporarily closed, Missing status or Observed operating

- Code:

In [7]:
status = chicago_restaurants["state"].astype("string").str.strip()
is_permanently_closed = status.str.casefold().eq("permanently closed").fillna(False)
is_temporarily_closed = status.str.contains(
    "temporarily closed", case=False, na=False
)
is_missing_status = status.isna() | status.eq("")
is_observed_operating = ~(
    is_permanently_closed | is_temporarily_closed | is_missing_status
)

chicago_restaurants["status_group"] = np.select(
    [
        is_permanently_closed,
        is_temporarily_closed,
        is_missing_status,
        is_observed_operating,
    ],
    [
        "Permanently closed",
        "Temporarily closed",
        "Missing status",
        "Observed operating",
    ],
    default="Unclassified",
)

print(chicago_restaurants["status_group"].value_counts())

status_group
Observed operating    3989
Missing status        2313
Permanently closed    1440
Temporarily closed     292
Name: count, dtype: int64


- Drop restaurants with labels "Missing status" and "Temporarily closed" since because they don't provide a defensible open vs. permanently-closed classification.

In [8]:
analysis_restaurants = chicago_restaurants.loc[
    chicago_restaurants["status_group"].isin(
        ["Observed operating", "Permanently closed"]
    ),
    [
        "gmap_id",
        "name",
        "address",
        "avg_rating",
        "num_of_reviews",
        "status_group",
    ],
].copy()
analysis_restaurants["permanently_closed"] = (
    analysis_restaurants["status_group"] == "Permanently closed"
).astype(int)

print("Number of rows after dropping 2 labels:", len(analysis_restaurants)) 

Number of rows after dropping 2 labels: 5429


## Dataset 2: Google Local Reviews

#### Step 1: Download and read dataset

Download the file review-Illinois.json.gz from Google Local Reviews Dataset - UCSD (https://cseweb.ucsd.edu/~jmcauley/datasets.html). Save it in EDA_Checkpoint/data/.

- Code:

In [9]:
google_reviews = pd.read_json(
    data_dir/"review-Illinois.json.gz",
    lines=True,
    compression="gzip",
    nrows=100000
)

#### Step 2: Filter

- No filter at this point

#### Step 3: Dropped Rows

- Check for duplicated or missing values, we just flag them

- Code:

In [10]:
print("Number of reviews with no text: ", google_reviews["text"].isna().sum())
print("Duplicate reviews:", google_reviews.duplicated(["user_id", "time"]).sum())

Number of reviews with no text:  29958
Duplicate reviews: 20204


#### Step 4: Type Fixes

- You can check for datatypes using .info() and summary statictics using .describe(). In this dataset, we will change'time' to correct format, it is now stores as milliseconds and we will covert it to show date and time

- Code

In [11]:
google_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   user_id  100000 non-null  float64
 1   name     100000 non-null  str    
 2   time     100000 non-null  int64  
 3   rating   100000 non-null  int64  
 4   text     70042 non-null   str    
 5   pics     2103 non-null    object 
 6   resp     22895 non-null   object 
 7   gmap_id  100000 non-null  str    
dtypes: float64(1), int64(2), object(2), str(3)
memory usage: 26.3+ MB


In [12]:
google_reviews.describe()

,user_id,time,rating
count,1.000000e+05,1.000000e+05,100000.000000
mean,1.093380e+20,1.561318e+12,4.421800
std,5.191733e+18,5.523083e+10,1.266563
min,1.000007e+20,1.201565e+12,1.000000
25%,1.049660e+20,1.527367e+12,5.000000
50%,1.090291e+20,1.570401e+12,5.000000
75%,1.137491e+20,1.607611e+12,5.000000
max,1.184467e+20,1.631143e+12,5.000000


In [13]:
google_reviews["time"] = pd.to_datetime(google_reviews["time"], unit = "ms")
print("Summary statistics for time after converting:\n", google_reviews["time"].describe())


Summary statistics for time after converting:
 count                        100000
mean     2019-06-23 19:19:25.804000
min             2008-01-29 00:00:00
25%      2018-05-26 20:37:20.873000
50%      2019-10-06 22:23:58.498000
75%      2020-12-10 14:42:38.782000
max      2021-09-08 23:17:20.232000
Name: time, dtype: object


## Dataset 3: Chicago Food Inspection

#### Step 1: Download and read dataset

- This dataset is pulled from Cicago Data Portal(https://data.cityofchicago.org/Health-Human-Services/Food-Inspections/4ijn-s7e5/about_data)

- Code:

In [14]:
food_url = (
    "https://data.cityofchicago.org/resource/"
    "4ijn-s7e5.csv?$limit=50000"
)

food_inspections = pd.read_csv(food_url)

#### Step 2: Filter to get only restaurants in Chicago

- Filter the dates we are focusing on (2018 - 2021), get only businesses that are retaurants  and only show the results that contain Pass, Pass w/ Conditions and Fail. Since each row represents one business in Chicago we don't have to filter only restaurants in Chicago like in previous datasets.

In [15]:
import difflib
import io
import re
import time
import unicodedata
import urllib.parse
import urllib.request
from collections import defaultdict


inspection_endpoint = "https://data.cityofchicago.org/resource/4ijn-s7e5.csv"
dollar = chr(36)
inspection_parameters = {
    dollar + "select": (
        "inspection_id,dba_name,address,inspection_date,results,latitude,longitude"
    ),
    dollar + "where": (
        "inspection_date between '2018-01-01T00:00:00' and '2021-12-31T23:59:59' "
        "AND facility_type='Restaurant' "
        "AND results in('Pass','Pass w/ Conditions','Fail')"
    ),
    dollar + "order": "inspection_date ASC, inspection_id ASC",
    dollar + "limit": "50000",
}
inspection_url = inspection_endpoint + "?" + urllib.parse.urlencode(
    inspection_parameters
)

with urllib.request.urlopen(inspection_url, timeout=120) as response: 
    food_inspections_historical = pd.read_csv(io.BytesIO(response.read()))

print("Inspections:", len(food_inspections_historical))

Inspections: 38401


#### Step 3: Dropped Rows

- Some rows in this dataset are missing inspection_date, name and location information such as longitude and latitude. Remove them and also remove rows that contain inspections outside of Chicago

In [16]:
print("Number of rows before dropping some of them:", len(food_inspections_historical))

Number of rows before dropping some of them: 38401


In [17]:
food_inspections_historical = food_inspections_historical.dropna(
    subset=["inspection_date", "latitude", "longitude", "dba_name"]
).copy()
food_inspections_historical = food_inspections_historical.loc[
    food_inspections_historical["latitude"].between(41.60, 42.10)
    & food_inspections_historical["longitude"].between(-87.95, -87.50)
].copy()
print("Number of rows after dropping some of them:", len(food_inspections_historical))

Number of rows after dropping some of them: 38360


#### Step 4: Type Fixes

- Ispection_date has a wrong data type and we need to sgange it from str to a date. You can check its datatype before and after you run the code using .info().

In [18]:
food_inspections_historical.info()

<class 'pandas.DataFrame'>
Index: 38360 entries, 0 to 38400
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   inspection_id    38360 non-null  int64  
 1   dba_name         38360 non-null  str    
 2   address          38360 non-null  str    
 3   inspection_date  38360 non-null  str    
 4   results          38360 non-null  str    
 5   latitude         38360 non-null  float64
 6   longitude        38360 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 4.9 MB


In [19]:
food_inspections_historical["inspection_date"] = pd.to_datetime(
    food_inspections_historical["inspection_date"],
    errors="coerce",
)

In [20]:
food_inspections_historical.info()

<class 'pandas.DataFrame'>
Index: 38360 entries, 0 to 38400
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   inspection_id    38360 non-null  int64         
 1   dba_name         38360 non-null  str           
 2   address          38360 non-null  str           
 3   inspection_date  38360 non-null  datetime64[us]
 4   results          38360 non-null  str           
 5   latitude         38360 non-null  float64       
 6   longitude        38360 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 4.0 MB


## Dataset 4: Princeton Business Data

Note: both Princeton datasets are now used as a replacement for our previous Yelp datasets. Since they are new and we didn't previosly inspect them in our EDA, these steps are a little different and maybe not as thorough as previous ones. Additionally, to access these datasets we needed a password from researchers at Princeton university which we are not going to share here.

#### Step 1: Download and read dataset

- This dataset is private from researchers at Princeton university. The dataset was too big so we kept the businesses only located in Chicago. The data can be viewed in Paraquet and is saved as businesses.parquet. Save it in EDA_Checkpoint/data/.

- Code:

In [21]:
#Princeton Business Data
princeton_businesses = pd.read_parquet(
    data_dir/"processed_princeton_chicago"/"businesses.parquet"
)

print("Princeton businesses:", len(princeton_businesses))

Princeton businesses: 5355


#### Step 2: Filter

- We already filtered it to get only businesses that are retaurants and are located in Chicago. You can inspect the data using .info(). Also, use .describe() for some important columns.

In [22]:
princeton_businesses.info()

<class 'pandas.DataFrame'>
RangeIndex: 5355 entries, 0 to 5354
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   business_id        5355 non-null   str    
 1   name               5355 non-null   str    
 2   address1           5352 non-null   str    
 3   city               5355 non-null   str    
 4   state              5355 non-null   str    
 5   postal_code        5355 non-null   str    
 6   latitude           5355 non-null   float64
 7   longitude          5355 non-null   float64
 8   category_aliases   5355 non-null   object 
 9   rating             5355 non-null   float64
 10  review_count       5355 non-null   int64  
 11  is_closed          5355 non-null   bool   
 12  source_partitions  5355 non-null   object 
 13  business_json      5355 non-null   str    
dtypes: bool(1), float64(3), int64(1), object(2), str(7)
memory usage: 5.4+ MB


In [23]:
princeton_businesses[['rating', 'is_closed', 'city']].head()

,rating,is_closed,city
0,5.0,False,Chicago
1,3.5,False,Chicago
2,3.5,False,Chicago
3,4.5,False,Chicago
4,4.0,False,Chicago


In [24]:
print(princeton_businesses["city"].value_counts())

city
Chicago    5353
CHICAGO       2
Name: count, dtype: int64


#### Step 3: Dropped Rows

- Check if there are any duplicate businesses based on their id, and if there are nay drop them

In [25]:
print("List number of duplicated businesses:", princeton_businesses.duplicated(subset = "business_id").sum())

List number of duplicated businesses: 0


#### Step 4: Type Fixes

- We noticed that city has 2 rows that are names CHICAGO and rest of them Chicago, so we need to change them to match. Additionaly, in order to join this dataset to our previous one we need to change is_closed. You can check its datatype before and after you run the code using .info().

- Code:

In [26]:
princeton_businesses.info()

<class 'pandas.DataFrame'>
RangeIndex: 5355 entries, 0 to 5354
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   business_id        5355 non-null   str    
 1   name               5355 non-null   str    
 2   address1           5352 non-null   str    
 3   city               5355 non-null   str    
 4   state              5355 non-null   str    
 5   postal_code        5355 non-null   str    
 6   latitude           5355 non-null   float64
 7   longitude          5355 non-null   float64
 8   category_aliases   5355 non-null   object 
 9   rating             5355 non-null   float64
 10  review_count       5355 non-null   int64  
 11  is_closed          5355 non-null   bool   
 12  source_partitions  5355 non-null   object 
 13  business_json      5355 non-null   str    
dtypes: bool(1), float64(3), int64(1), object(2), str(7)
memory usage: 5.4+ MB


In [27]:
princeton_businesses["city"] = princeton_businesses["city"].str.title()
princeton_businesses["is_closed"] = princeton_businesses["is_closed"].astype(int)
princeton_businesses["city"].value_counts()


city
Chicago    5355
Name: count, dtype: int64

In [28]:
princeton_businesses["is_closed"].value_counts()

is_closed
0    5323
1      32
Name: count, dtype: int64

In [29]:
princeton_businesses.info()

<class 'pandas.DataFrame'>
RangeIndex: 5355 entries, 0 to 5354
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   business_id        5355 non-null   str    
 1   name               5355 non-null   str    
 2   address1           5352 non-null   str    
 3   city               5355 non-null   str    
 4   state              5355 non-null   str    
 5   postal_code        5355 non-null   str    
 6   latitude           5355 non-null   float64
 7   longitude          5355 non-null   float64
 8   category_aliases   5355 non-null   object 
 9   rating             5355 non-null   float64
 10  review_count       5355 non-null   int64  
 11  is_closed          5355 non-null   int64  
 12  source_partitions  5355 non-null   object 
 13  business_json      5355 non-null   str    
dtypes: float64(3), int64(2), object(2), str(7)
memory usage: 5.4+ MB


## Dataset 5: Princeton Review Data

#### Step 1: Download and read dataset

- This dataset is private from researchers at Princeton university. The dataset was too big so we kept the businesses only located in Chicago. The data can be viewed in Paraquet and is saved as reviews.parquet. Save it in EDA_Checkpoint/data/.

In [30]:
princeton_reviews = pd.read_parquet( 
    data_dir/"processed_princeton_chicago"/"reviews.parquet" )

print("Princeton reviews:", len(princeton_reviews))

Princeton reviews: 1331020


#### Step 2: Filter

We already filtered it to get only businesses that are retaurants and are located in Chicago. You can inspect the data using .info(). Also, use .describe() for some important columns.

In [31]:
princeton_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 1331020 entries, 0 to 1331019
Data columns (total 13 columns):
 #   Column                 Non-Null Count    Dtype 
---  ------                 --------------    ----- 
 0   business_id            1331020 non-null  str   
 1   review_key             1331020 non-null  str   
 2   source_partition       1331020 non-null  str   
 3   recommendation_status  1331020 non-null  str   
 4   content                1331020 non-null  str   
 5   date                   1331020 non-null  str   
 6   rating                 1331020 non-null  int64 
 7   elite                  1169366 non-null  object
 8   user_friends           1331020 non-null  str   
 9   user_review_count      1331020 non-null  str   
 10  user_photos            1331020 non-null  str   
 11  data_hovercard_id      161654 non-null   str   
 12  other_fields_json      0 non-null        str   
dtypes: int64(1), object(1), str(11)
memory usage: 986.4+ MB


In [32]:
princeton_reviews[['date', 'recommendation_status', 'user_review_count']].head()

,date,recommendation_status,user_review_count
0,7/28/2021,recommended,217
1,9/23/2021,recommended,190
2,9/19/2021,recommended,375
3,8/22/2021,recommended,2
4,7/26/2021,recommended,303


#### Step 3: Dropped Rows

- Check if there are any duplicate reviews based on their key, and if there are nay drop them. Additionally check if any columns are missing a lot of info.

In [33]:
print("List number of duplicated reviews:", princeton_reviews.duplicated(subset = "review_key").sum())

List number of duplicated reviews: 0


In [34]:
princeton_reviews.isnull().sum()

business_id                    0
review_key                     0
source_partition               0
recommendation_status          0
content                        0
date                           0
rating                         0
elite                     161654
user_friends                   0
user_review_count              0
user_photos                    0
data_hovercard_id        1169366
other_fields_json        1331020
dtype: int64

In [35]:
princeton_reviews['other_fields_json'].info()

<class 'pandas.Series'>
RangeIndex: 1331020 entries, 0 to 1331019
Series name: other_fields_json
Non-Null Count  Dtype
--------------  -----
0 non-null      str  
dtypes: str(1)
memory usage: 10.3 MB


In [36]:
princeton_reviews['data_hovercard_id'].info()

<class 'pandas.Series'>
RangeIndex: 1331020 entries, 0 to 1331019
Series name: data_hovercard_id
Non-Null Count   Dtype
--------------   -----
161654 non-null  str  
dtypes: str(1)
memory usage: 13.7 MB


In [37]:
princeton_reviews = princeton_reviews.drop(columns = ["other_fields_json"])

In [38]:
princeton_reviews.isnull().sum()

business_id                    0
review_key                     0
source_partition               0
recommendation_status          0
content                        0
date                           0
rating                         0
elite                     161654
user_friends                   0
user_review_count              0
user_photos                    0
data_hovercard_id        1169366
dtype: int64

#### Step 4: Type Fixes

In [39]:
 princeton_reviews.info() 

<class 'pandas.DataFrame'>
RangeIndex: 1331020 entries, 0 to 1331019
Data columns (total 12 columns):
 #   Column                 Non-Null Count    Dtype 
---  ------                 --------------    ----- 
 0   business_id            1331020 non-null  str   
 1   review_key             1331020 non-null  str   
 2   source_partition       1331020 non-null  str   
 3   recommendation_status  1331020 non-null  str   
 4   content                1331020 non-null  str   
 5   date                   1331020 non-null  str   
 6   rating                 1331020 non-null  int64 
 7   elite                  1169366 non-null  object
 8   user_friends           1331020 non-null  str   
 9   user_review_count      1331020 non-null  str   
 10  user_photos            1331020 non-null  str   
 11  data_hovercard_id      161654 non-null   str   
dtypes: int64(1), object(1), str(10)
memory usage: 976.0+ MB


- Some columns have different data types that should be fixed and changed such as user_friend, user_review_count, user_photos

In [40]:
princeton_reviews["user_friends"] = princeton_reviews["user_friends"].astype(int)
princeton_reviews["user_review_count"] = princeton_reviews["user_review_count"].astype(int)
princeton_reviews["user_photos"] = princeton_reviews["user_photos"].astype(int)
princeton_reviews["date"] = pd.to_datetime(princeton_reviews["date"], format = "%m/%d/%Y", errors = "coerce")
princeton_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 1331020 entries, 0 to 1331019
Data columns (total 12 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   business_id            1331020 non-null  str           
 1   review_key             1331020 non-null  str           
 2   source_partition       1331020 non-null  str           
 3   recommendation_status  1331020 non-null  str           
 4   content                1331020 non-null  str           
 5   date                   1299180 non-null  datetime64[us]
 6   rating                 1331020 non-null  int64         
 7   elite                  1169366 non-null  object        
 8   user_friends           1331020 non-null  int64         
 9   user_review_count      1331020 non-null  int64         
 10  user_photos            1331020 non-null  int64         
 11  data_hovercard_id      161654 non-null   str           
dtypes: datetime64[us](1), int64(4), object(

## Dataset 6: Chicago Business Licences

- This dataset is secondary dataset and we haven't used it in our analasys yet. This dataset could give each restaurant an age and a rough closure date and might be used for one of our reserch questions 'Is review volume still associated with permanent closure after accounting for restaurant age, category, and chain status?' After adding new primary datasets we need some time to decide whether to use it and we will add more info about it if so.

## Joins

Note: most of the code was reused from EDA

In [41]:
import difflib
import io
import re
import time
import unicodedata
import urllib.parse
import urllib.request
from collections import defaultdict


def normalize_business_name(value):
    normalized = (
        unicodedata.normalize("NFKD", str(value))
        .encode("ascii", "ignore")
        .decode()
    )
    normalized = re.sub(r"[^a-z0-9 ]+", " ", normalized.lower())
    normalized = re.sub(
        r"\b(inc|llc|corp|corporation|company|co|restaurant|restaurants|the)\b",
        " ",
        normalized,
    )
    return " ".join(normalized.split())


def extract_street_number(value):
    match = re.search(r"\b(\d+)\b", str(value))
    return match.group(1) if match else None

def haversine_distance_m(lat1, lon1, lat2, lon2):
    radius_m = 6_371_000.0
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    haversine = (
        np.sin(delta_phi / 2) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2) ** 2
    )
    return 2 * radius_m * np.arcsin(np.sqrt(haversine))

def match_inspections(businesses, id_col, name_col, address_col):
    match_table = businesses[[id_col, name_col, address_col, "latitude", "longitude"]].dropna(
        subset = ["latitude", "longitude"]).copy()

    match_table["normalized_name"] = match_table[name_col].map(
        normalize_business_name
    )
    match_table["street_number"] = match_table[address_col].map(
        extract_street_number
    )
    match_table["grid_lat"] = np.floor(match_table["latitude"] * 1000).astype(int)
    match_table["grid_lon"] = np.floor(match_table["longitude"] * 1000).astype(int)

    inspection_match_table = food_inspections_historical.copy()
    inspection_match_table["normalized_name"] = inspection_match_table["dba_name"].map(normalize_business_name)
    inspection_match_table["street_number"] = inspection_match_table["address"].map(extract_street_number)
    inspection_match_table["grid_lat"] = np.floor(
    inspection_match_table["latitude"] * 1000).astype(int)
    inspection_match_table["grid_lon"] = np.floor(inspection_match_table["longitude"] * 1000).astype(int)

    spatial_grid = defaultdict(list)
    for google_index, google_row in match_table.iterrows():
        spatial_grid[(google_row.grid_lat, google_row.grid_lon)].append(google_index)
    
    candidate_matches = []
    for inspection in inspection_match_table.itertuples(index=False):
        candidate_indices = []
        for latitude_offset in (-1, 0, 1):
            for longitude_offset in (-1, 0, 1):
                candidate_indices.extend(
                    spatial_grid.get(
                        (
                            inspection.grid_lat + latitude_offset,
                            inspection.grid_lon + longitude_offset,
                        ),
                        [],
                    )
                )
        if not candidate_indices:
            continue
        nearby = match_table.loc[candidate_indices]
        distances = haversine_distance_m(
            inspection.latitude,
            inspection.longitude,
            nearby["latitude"].to_numpy(),
            nearby["longitude"].to_numpy(),
        )

        
        for google_index, distance_m in zip(nearby.index, distances):
            if distance_m > 60:
                continue
            google_candidate = match_table.loc[google_index]
            name_similarity = difflib.SequenceMatcher(
                None,
                inspection.normalized_name,
                google_candidate.normalized_name,
            ).ratio()
            same_street_number = bool(
                inspection.street_number
                and google_candidate.street_number
                and inspection.street_number == google_candidate.street_number
            )
            name_containment = bool(
                min(
                    len(inspection.normalized_name),
                    len(google_candidate.normalized_name),
                )
                >= 4
                and (
                    inspection.normalized_name in google_candidate.normalized_name
                    or google_candidate.normalized_name in inspection.normalized_name
                )
            )

            strong_name_match = name_similarity >= 0.80 and distance_m <= 60
            address_supported_match = (
                distance_m <= 40
                and same_street_number
                and (name_similarity >= 0.55 or name_containment)
            )
            if strong_name_match or address_supported_match:
                candidate_matches.append(
                    {
                        "inspection_id": inspection.inspection_id,
                        "inspection_name": inspection.dba_name,
                        "inspection_date": inspection.inspection_date,
                        "results": inspection.results,
                         id_col: google_candidate[id_col],
                        "restaurant_name": google_candidate[name_col],
                        "distance_m": float(distance_m),
                        "name_similarity": name_similarity,
                        "match_score": name_similarity - float(distance_m) / 1000,
                    }
                )

    matched_inspections = pd.DataFrame(candidate_matches)
    matched_inspections = (
        matched_inspections.sort_values(
            ["inspection_id", "match_score"],
            ascending=[True, False],
        )
        .drop_duplicates("inspection_id")
        .reset_index(drop=True)
    ) 

    return matched_inspections

In [42]:
google_matched_inspections = match_inspections(chicago_restaurants, id_col = "gmap_id", name_col = "name", address_col = "address")
print("Number of matched inspections:", len(google_matched_inspections))
print("Number of unmatched inspections:", len(food_inspections_historical)-len(google_matched_inspections))
print("Number of matched restaurants:", google_matched_inspections["gmap_id"].nunique())
print("Number of unmatched restaurants:", len(chicago_restaurants)-google_matched_inspections["gmap_id"].nunique())


Number of matched inspections: 22831
Number of unmatched inspections: 15529
Number of matched restaurants: 5182
Number of unmatched restaurants: 2852


In [43]:
princeton_matched_inspections = match_inspections(princeton_businesses, id_col = "business_id", name_col = "name", address_col = "address1")
print("Number of matched inspections:", len(princeton_matched_inspections))
print("Number of unmatched inspections:", len(food_inspections_historical)-len(princeton_matched_inspections))
print("Number of matched restaurants:", princeton_matched_inspections["business_id"].nunique())
print("Number of unmatched restaurants:", len(princeton_businesses) - princeton_matched_inspections["business_id"].nunique())

Number of matched inspections: 17789
Number of unmatched inspections: 20571
Number of matched restaurants: 3771
Number of unmatched restaurants: 1584


- We are joining Google Local Business Metadata with Chicago Food Inspections and Princeton businesses with Chicago Food Inspections
- Keys: these datasets don't share common ID so they are being matched from neighboring latitude and longitude, address and name. We accept the match of restaurant and inspections if: name similarity is at least 0.80 within 60 meter or the street number agrees within 40 meters and the names have some moderate similarity.
- For the first join (Google Local Business Metadata with Chicago Food Inspections) there are 22831 matched isnpections and 15529 unmatched. Additionally, there are 5283 matched restaurants and 2852 unmatched.
- For the second join (Princeton businesses with Chicago Food Inspections) there are 17789 matched isnpections and 20571 unmatched. Additionally, there are 3771 matched restaurants and 1584 unmatched.